# Encontro 3 · Dentro do RAG

**Pergunta do dia:** o que muda quando o agente decide *se* e *quando* buscar?

Antes de o agente decidir buscar, a busca precisa trazer o trecho certo. Este notebook abre a busca por dentro, usando a wiki da Aurora. A wiki cresceu desde o Encontro 1: as mesmas 12 páginas, agora com seções, mais as páginas do Financeiro e as que vieram na migração do Confluence antigo. Ela tem quatro armadilhas, as mesmas que aparecem em empresas de verdade:

1. uma regra cuja **exceção** está no parágrafo seguinte;
2. **duas versões** da mesma política, a de 2024 e a de 2026;
3. perguntas em que só um **termo exato** (um código, um número de erro) separa a resposta certa da errada;
4. perguntas que a wiki **não responde**.

**Como usar.** Siga as etapas na ordem e rode as células de cima para baixo. Cada etapa diz no topo o que ela produz e termina com uma pergunta para o grupo responder antes de seguir. As caixas *Para ir além* são opcionais: deixe para depois.

**O entregável do dia está no notebook `02-hands-on-chunking.ipynb`.** Aqui não há nada a entregar.

## Etapa 0 · Ambiente

**Produz:** a confirmação de que tudo está instalado e o modelo de embeddings baixou.

Na primeira vez, a célula abaixo baixa cerca de 470 MB do Hugging Face e pode levar alguns minutos. Se der erro, rode `pip install -r requirements.txt` no terminal e tente de novo.

In [1]:
import rag_aurora as r

emb = r.Embeddings()
teste = emb.consultas(["teste"])
print(f"Modelo: {emb.nome} · vetores de {teste.shape[1]} dimensões")
print("Tudo certo. Pode seguir.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo: intfloat/multilingual-e5-small · vetores de 384 dimensões
Tudo certo. Pode seguir.


## Etapa 1 · O corpus

**Produz:** `docs`, as 25 páginas da wiki, cada uma com texto e metadados.

In [2]:
docs = r.carregar_corpus()
print(len(docs), "páginas\n")
for espaco in sorted({d.metadata["espaco"] for d in docs}):
    ids = [d.metadata["id"] for d in docs if d.metadata["espaco"] == espaco]
    print(f"{espaco:<18} {len(ids):>2}  {', '.join(ids)}")

25 páginas

confluence-rh       2  rh-home-office-2024, rh-organograma-2024
wiki-beneficios     6  beneficios-auxilio-creche, beneficios-bem-estar, beneficios-plano-de-saude, beneficios-plano-odontologico, beneficios-seguro-de-vida, beneficios-vale-refeicao
wiki-financeiro     3  fin-adiantamento-de-viagem, fin-cartao-corporativo, fin-viagens-e-reembolso
wiki-rh             8  rh-avaliacao-desempenho, rh-codigo-de-conduta, rh-desligamento, rh-ferias, rh-guia-de-integracao, rh-jornada-e-banco-de-horas, rh-licencas, rh-trabalho-remoto
wiki-ti             6  ti-acesso-a-sistemas, ti-equipamentos, ti-instalacao-de-software, ti-senhas-e-acesso, ti-vpn-codigos-de-erro, ti-vpn-e-acesso-remoto


In [3]:
# O cabeçalho de cada página vira metadado. Repare em vigencia, politica e autoridade.
r.pagina(docs, "rh-home-office-2024").metadata

{'id': 'rh-home-office-2024',
 'titulo': 'Política de Home Office',
 'espaco': 'confluence-rh',
 'tema': 'rh',
 'politica': 'trabalho-remoto',
 'dono': 'RH Corporativo',
 'autoridade': 'oficial',
 'vigencia': '2024-01-15',
 'atualizado_em': '2024-01-10',
 'arquivo': 'rh-home-office-2024.md'}

**Pergunta para o grupo.** Olhando só os metadados, que campo você usaria para descobrir que `rh-home-office-2024` e `rh-trabalho-remoto` tratam do mesmo assunto? E qual diria qual delas vale hoje?

## Etapa 2 · Chunking

**Produz:** `chunks_c` (corte por tamanho) e `chunks_e` (corte pela estrutura do documento).

O corte por tamanho é o padrão da maioria dos tutoriais: blocos de até 400 caracteres. O corte por estrutura usa as seções (`##`) que o autor da política já escreveu.

In [4]:
chunks_c = r.chunks_por_caracteres(docs, tamanho=400, sobreposicao=0)
chunks_e = r.chunks_por_estrutura(docs)
print("por tamanho:  ", r.resumo_chunks(chunks_c))
print("por estrutura:", r.resumo_chunks(chunks_e))

por tamanho:   64 chunks · tamanho mínimo 95, mediano 304, máximo 395 caracteres
por estrutura: 90 chunks · tamanho mínimo 66, mediano 194, máximo 651 caracteres


In [5]:
# A política de viagens cortada por tamanho. Procure a seção 5, Alimentação.
r.mostrar_chunks(chunks_c, "fin-viagens-e-reembolso")

── fin-viagens-e-reembolso#0 (290 caracteres) ──────────────────────────────
# Viagens a trabalho e reembolso de despesas

## 1. Aprovação da viagem

1.1 Toda viagem a trabalho precisa ser aprovada pela liderança no Portal de Viagens antes da compra de passagens e hotel.

1.2 Viagens internacionais também precisam da aprovação da diretoria da área.

## 2. Passagens

── fin-viagens-e-reembolso#1 (395 caracteres) ──────────────────────────────
2.1 Passagens aéreas são compradas pela agência credenciada, em classe econômica, com pelo menos 10 dias de antecedência sempre que possível.

## 3. Transporte local

3.1 Táxi e aplicativo de transporte são reembolsáveis no trajeto entre aeroporto, hotel e local de trabalho.

3.2 Quilometragem com carro próprio é reembolsada a R$ 1,20 por km, mediante registro do trajeto no Portal de Viagens.

── fin-viagens-e-reembolso#2 (270 caracteres) ──────────────────────────────
## 4. Prestação de contas

4.1 As despesas devem ser lançadas no Portal de Viage

In [6]:
# A mesma política cortada pela estrutura.
r.mostrar_chunks(chunks_e, "fin-viagens-e-reembolso")

── fin-viagens-e-reembolso#0 (269 caracteres) ──────────────────────────────
Viagens a trabalho e reembolso de despesas > 1. Aprovação da viagem
1.1 Toda viagem a trabalho precisa ser aprovada pela liderança no Portal de Viagens antes da compra de passagens e hotel.  
1.2 Viagens internacionais também precisam da aprovação da diretoria da área.

── fin-viagens-e-reembolso#1 (199 caracteres) ──────────────────────────────
Viagens a trabalho e reembolso de despesas > 2. Passagens
2.1 Passagens aéreas são compradas pela agência credenciada, em classe econômica, com pelo menos 10 dias de antecedência sempre que possível.

── fin-viagens-e-reembolso#2 (294 caracteres) ──────────────────────────────
Viagens a trabalho e reembolso de despesas > 3. Transporte local
3.1 Táxi e aplicativo de transporte são reembolsáveis no trajeto entre aeroporto, hotel e local de trabalho.  
3.2 Quilometragem com carro próprio é reembolsada a R$ 1,20 por km, mediante registro do trajeto no Portal de Viagens.

─

**Pergunta para o grupo.** Um agente que recebesse só o chunk `fin-viagens-e-reembolso#3` do corte por tamanho, o que responderia para quem vai dormir em Brasília? E onde ficou o título "5. Alimentação"?

> **Para ir além (opcional).** Rode de novo o corte por tamanho com `tamanho=800, sobreposicao=100` e veja se a seção 5 fica inteira. Depois rode o corte por estrutura com `prefixar_titulo=False` e leia o chunk da cláusula 5.3: dá para saber de que política ele é?

## Etapa 3 · Embeddings na mão

**Produz:** a intuição do que a busca vetorial mede, e do que ela não mede.

Um embedding transforma um texto num vetor. Textos com sentido parecido ficam com vetores próximos. A proximidade é medida pelo cosseno: 1 é idêntico, perto de 0 é sem relação.

In [7]:
frases = [
    "Posso trabalhar de casa três dias por semana.",
    "O home office é permitido até três dias na semana.",
    "Erro 809 na VPN: ative o Modo TCP 443.",
    "Erro 812 na VPN: peça inclusão no grupo de acesso remoto.",
]
v = emb.passagens(frases)
for i, a in enumerate(frases):
    print(f"{i}  " + "  ".join(f"{r.similaridade(v[i], v[j]):.2f}" for j in range(len(frases))) + f"   {a}")

0  1.00  0.93  0.84  0.83   Posso trabalhar de casa três dias por semana.
1  0.93  1.00  0.84  0.84   O home office é permitido até três dias na semana.
2  0.84  0.84  1.00  0.91   Erro 809 na VPN: ative o Modo TCP 443.
3  0.83  0.84  0.91  1.00   Erro 812 na VPN: peça inclusão no grupo de acesso remoto.


As frases 0 e 1 dizem a mesma coisa com palavras diferentes, e ficam próximas. As frases 2 e 3 dizem coisas **opostas** sobre a VPN e também ficam próximas, porque têm quase o mesmo formato. Guarde isso para a Etapa 5.

Agora, a pergunta mais comum da wiki contra as duas políticas de trabalho remoto:

In [8]:
pergunta = "Quantos dias de home office posso fazer por semana?"
q = emb.consultas([pergunta])[0]
for c in chunks_e:
    if c.metadata["politica"] == "trabalho-remoto" and "1." in c.metadata.get("secao", ""):
        p = emb.passagens([c.page_content])[0]
        print(f"{r.similaridade(q, p):.3f}  {c.metadata['id']:<26} vigência {c.metadata['vigencia']}")

0.939  rh-home-office-2024        vigência 2024-01-15
0.873  rh-trabalho-remoto         vigência 2026-02-10


**Pergunta para o grupo.** A política de 2024, que não vale mais, ficou mais parecida com a pergunta? Se ficou, a busca errou? O que a similaridade mede, e o que ela não tem como saber?

> **Para ir além (opcional).** A família E5 foi treinada com os prefixos `query:` e `passage:`. Calcule a mesma similaridade com `emb.modelo.encode([...], normalize_embeddings=True)`, sem os prefixos, e compare.

## Etapa 4 · Indexar e medir

**Produz:** dois índices no Chroma e o primeiro placar.

O placar usa o gabarito `casos.json`: para cada pergunta, o trecho que precisa aparecer entre os 3 primeiros resultados (**achou**). Também marca quando um trecho enganoso, como a versão velha, o guia que resume outra página ou a regra sem a exceção, aparece antes do certo (**contaminou**).

Esta etapa não chama nenhum LLM. É busca pura, então é barata e dá sempre o mesmo resultado.

In [9]:
# Travou nas etapas anteriores? Descomente as duas linhas abaixo e rode.
# globals().update(r.checkpoint())
# casos = r.carregar_casos()

In [10]:
casos = r.carregar_casos()
vet_c = r.IndiceVetorial(chunks_c, emb)
vet_e = r.IndiceVetorial(chunks_e, emb)

experimentos = []
r.registrar(experimentos, "vetor · tamanho 400", r.avaliar(vet_c.buscar, casos, k=3))
r.registrar(experimentos, "vetor · estrutura", r.avaliar(vet_e.buscar, casos, k=3))
r.comparar(experimentos)

configuração                normal      excecao     conflito  termo-exato   autoridade       TOTAL
vetor · tamanho 400        7/7 (0)      2/2 (1)      2/3 (3)      3/3 (1)      1/1 (0)   15/16 (5)
vetor · estrutura          6/7 (0)      2/2 (0)      2/3 (3)      3/3 (0)      1/1 (0)   14/16 (3)

formato: achou/casos (contaminados). Melhor = achou alto e contaminados zero.


In [11]:
# Como olhar um caso de perto: a pergunta e o que voltou.
caso = next(c for c in casos if c["id"] == "E1")
print(caso["pergunta"], "\n")
print("── por tamanho"); r.mostrar(vet_c.buscar(caso["pergunta"], 3))
print("\n── por estrutura"); r.mostrar(vet_e.buscar(caso["pergunta"], 3))

Qual o limite diário de alimentação numa viagem com pernoite em Brasília? 

── por tamanho
1. [0.902] fin-viagens-e-reembolso#4  ·  wiki-financeiro · vigência 2026-01-01 · oficial
   5.3 Esse teto não vale quando o pernoite for em São Paulo, Rio de Janeiro ou Brasília, cidades classificadas como de custo elevado. Nelas o limite diário passa …
2. [0.884] fin-viagens-e-reembolso#3  ·  wiki-financeiro · vigência 2026-01-01 · oficial
   5.1 As refeições durante a viagem são reembolsadas mediante nota fiscal, inclusive café da manhã quando não estiver incluído na diária do hotel. 5.2 O limite di…
3. [0.852] fin-viagens-e-reembolso#5  ·  wiki-financeiro · vigência 2026-01-01 · oficial
   6.1 A hospedagem é reservada pela agência credenciada, em hotel da lista de parceiros, com diária de até R$ 450,00 em capitais e R$ 320,00 nas demais cidades. 6…

── por estrutura
1. [0.905] fin-viagens-e-reembolso#4  ·  wiki-financeiro · vigência 2026-01-01 · oficial
   Viagens a trabalho e reembolso de des

**Pergunta para o grupo.** Em que categorias a estrutura ganhou do tamanho? Em quais **nenhuma** das duas resolveu? Guarde as que não resolveram: são as próximas etapas.

> **Para ir além (opcional).** `r.falhas(r.avaliar(vet_e.buscar, casos, k=3))` lista os casos que falharam. Escolha um e olhe o que voltou com `r.mostrar(...)`.

## Etapa 5 · Termo exato

**Produz:** a comparação entre busca vetorial, léxica (BM25) e híbrida.

O BM25 não entende sentido nenhum. Ele conta palavras em comum e dá mais peso às palavras raras, como "809" e "AUR-S2". A busca híbrida junta as duas listas por *Reciprocal Rank Fusion*: cada lista vota pela posição de cada chunk, sem comparar as notas, que estão em escalas diferentes.

In [12]:
lex_e = r.IndiceLexico(chunks_e)
pergunta = "Estou com o erro 809 na VPN, o que eu faço?"
print("── vetorial"); r.mostrar(vet_e.buscar(pergunta, 3))
print("\n── BM25");    r.mostrar(lex_e.buscar(pergunta, 3))

── vetorial
1. [0.889] ti-vpn-codigos-de-erro#3  ·  wiki-ti · vigência 2026-02-03 · oficial
   Códigos de erro da VPN > Erro 809 A conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de hotel e de aeroporto.…
2. [0.886] ti-vpn-codigos-de-erro#1  ·  wiki-ti · vigência 2026-02-03 · oficial
   Códigos de erro da VPN > Erro 691 A conexão com a VPN foi recusada porque o usuário ou a senha não foram aceitos. Confirme se a senha não expirou e tente novame…
3. [0.874] ti-vpn-codigos-de-erro#4  ·  wiki-ti · vigência 2026-02-03 · oficial
   Códigos de erro da VPN > Erro 812 A conexão com a VPN foi recusada porque o seu usuário não pertence a um grupo com acesso remoto. Peça a inclusão no grupo de a…

── BM25
1. [11.3] ti-vpn-codigos-de-erro#3  ·  wiki-ti · vigência 2026-02-03 · oficial
   Códigos de erro da VPN > Erro 809 A conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de hotel e de aeroporto.…


In [13]:
hibrida = lambda pergunta, k: r.busca_hibrida(vet_e, lex_e, pergunta, k)
r.registrar(experimentos, "BM25 · estrutura", r.avaliar(lex_e.buscar, casos, k=3))
r.registrar(experimentos, "híbrida · estrutura", r.avaliar(hibrida, casos, k=3))
r.comparar(experimentos)

configuração                normal      excecao     conflito  termo-exato   autoridade       TOTAL
vetor · tamanho 400        7/7 (0)      2/2 (1)      2/3 (3)      3/3 (1)      1/1 (0)   15/16 (5)
vetor · estrutura          6/7 (0)      2/2 (0)      2/3 (3)      3/3 (0)      1/1 (0)   14/16 (3)
BM25 · estrutura           5/7 (0)      2/2 (0)      1/3 (3)      3/3 (0)      1/1 (0)   12/16 (3)
híbrida · estrutura        6/7 (0)      2/2 (0)      1/3 (3)      3/3 (0)      1/1 (0)   13/16 (3)

formato: achou/casos (contaminados). Melhor = achou alto e contaminados zero.


**Pergunta para o grupo.** O BM25 sozinho perde onde? A híbrida perde alguma coisa em relação ao vetor? Em que tipo de pergunta da *sua* empresa o termo exato manda: código de produto, número de contrato, nome de sistema?

## Etapa 6 · Quando a wiki não sabe

**Produz:** a prova de que a busca nunca devolve "nada".

In [14]:
for caso in casos:
    if caso["categoria"] in ("sem-resposta", "normal"):
        melhor = vet_e.buscar(caso["pergunta"], 1)[0]
        print(f"{melhor[1]:.3f}  {caso['categoria']:<13} {caso['pergunta']}")

0.925  normal        Quantos dias de licença-paternidade eu tenho?
0.897  normal        Com quanta antecedência preciso pedir minhas férias?
0.920  normal        Posso dividir minhas férias em mais de um período?
0.879  normal        Em quanto tempo a TI conserta ou repõe um notebook quebrado?
0.874  normal        Roubaram meu notebook do trabalho. O que eu faço?
0.907  normal        Qual o valor máximo do auxílio-creche?
0.875  normal        Quantas sessões de terapia a empresa paga?
0.867  sem-resposta  A Aurora oferece previdência privada?
0.829  sem-resposta  Qual é a política de estacionamento do escritório?
0.843  sem-resposta  A empresa paga parte da minha pós-graduação?


A busca sempre devolve os k chunks mais parecidos, mesmo quando nenhum serve. A nota do melhor resultado é menor nas perguntas sem resposta, mas as duas faixas costumam se sobrepor.

**Pergunta para o grupo.** Em que nota você cortaria para o agente dizer "não sei"? O corte erraria algum caso normal? Se um corte fixo não separa bem, onde mais o "não sei" pode ser projetado? (Dica: no que o agente faz com o trecho, não no retriever.)

## Etapa 7 · Conflito e autoridade

**Produz:** a configuração final, com metadados resolvendo o que o modelo não resolve.

Duas correções, para dois problemas diferentes:

- **Versão velha.** Para cada `politica`, fica só a página com a `vigencia` mais recente (`so_a_versao_vigente`).
- **Fonte sem autoridade.** É o c07 dos Encontros 1 e 2: o guia de integração diz que o vale-refeição é R$ 42, e a página de Benefícios diz R$ 45. Lá, a resposta certa era "não sei". Agora o guia está marcado como `autoridade: resumo`, porque repete regras de outras páginas, e o filtro `autoridade = oficial` o tira da disputa.

In [15]:
pergunta = "Quantos dias de home office posso fazer por semana?"
print("── híbrida, sem tratar versões"); r.mostrar(hibrida(pergunta, 3))
print("\n── só a versão vigente");       r.mostrar(r.so_a_versao_vigente(hibrida(pergunta, 9))[:3])

── híbrida, sem tratar versões
1. [0.0328] rh-home-office-2024#1  ·  confluence-rh · vigência 2024-01-15 · oficial
   Política de Home Office > 1. Quantos dias de home office posso fazer 1.1 Todo colaborador pode fazer home office até 2 dias por semana. Os dias de home office s…
2. [0.0323] rh-home-office-2024#3  ·  confluence-rh · vigência 2024-01-15 · oficial
   Política de Home Office > 3. Home office todos os dias 3.1 Home office em todos os dias da semana não é permitido, salvo em casos médicos com laudo.
3. [0.0313] rh-home-office-2024#2  ·  confluence-rh · vigência 2024-01-15 · oficial
   Política de Home Office > 2. Auxílio home office 2.1 Quem faz home office recebe um auxílio home office de R$ 80,00 por mês para ajudar com internet e energia.

── só a versão vigente
1. [0.0304] rh-guia-de-integracao#1  ·  wiki-rh · vigência 2026-01-15 · resumo
   Guia de integração para novos colaboradores > Modelo de trabalho Somos híbridos. Você pode trabalhar de casa até 3 dias por semana,

In [16]:
def final(pergunta, k):
    candidatos = r.busca_hibrida(vet_e, lex_e, pergunta, k * 3, filtro={"autoridade": "oficial"})
    return r.so_a_versao_vigente(candidatos)[:k]

r.registrar(experimentos, "final: híbrida + vigente + oficial", r.avaliar(final, casos, k=3))
r.comparar(experimentos)

configuração                               normal      excecao     conflito  termo-exato   autoridade       TOTAL
vetor · tamanho 400                       7/7 (0)      2/2 (1)      2/3 (3)      3/3 (1)      1/1 (0)   15/16 (5)
vetor · estrutura                         6/7 (0)      2/2 (0)      2/3 (3)      3/3 (0)      1/1 (0)   14/16 (3)
BM25 · estrutura                          5/7 (0)      2/2 (0)      1/3 (3)      3/3 (0)      1/1 (0)   12/16 (3)
híbrida · estrutura                       6/7 (0)      2/2 (0)      1/3 (3)      3/3 (0)      1/1 (0)   13/16 (3)
final: híbrida + vigente + oficial        6/7 (0)      2/2 (0)      3/3 (0)      3/3 (0)      1/1 (0)   15/16 (0)

formato: achou/casos (contaminados). Melhor = achou alto e contaminados zero.


**Pergunta para o grupo.** Busca-se `k * 3` candidatos e só depois se filtra a versão. Por quê? E quem garante que os campos `vigencia` e `autoridade` estão certos em todas as páginas? No Encontro 1 o agente tinha de dizer "não sei" sobre o vale-refeição. O que mudou: o modelo, ou o dado?

> **Para ir além (opcional).** Remova a página `rh-home-office-2024.md` da pasta `corpus` e rode o placar sem o tratamento de versões. Apagar resolve? E o que acontece com quem precisa saber a regra de 2024 para um pedido feito em 2024?

## Etapa 7b · A web como fonte (demonstração do professor)

**Produz:** a mesma pergunta respondida pela wiki e pela web.

Esta etapa é demonstrada pelo professor. Ela usa a busca na web do Encontro 2 (API do Serper) e precisa de `SERPER_API_KEY` no `.env`. Sem a chave, a célula avisa, e você pode seguir para a Etapa 8.

Repare: isto também é RAG. Há recuperação e há geração aumentada, mas não há banco vetorial, nem chunking, nem metadado que a gente controle.

In [17]:
pergunta = "Quantos dias de home office posso fazer por semana?"
print("── wiki da Aurora (configuração final)"); r.mostrar(final(pergunta, 3))
print("\n── web (Serper)");                       r.mostrar_web(r.buscar_na_web(pergunta))

── wiki da Aurora (configuração final)
1. [0.0306] rh-trabalho-remoto#1  ·  wiki-rh · vigência 2026-02-10 · oficial
   Política de trabalho remoto > 1. Dias de trabalho remoto 1.1 Cada colaborador pode trabalhar remotamente até 3 dias por semana. 1.2 A quarta-feira é o dia prese…
2. [0.0305] rh-trabalho-remoto#4  ·  wiki-rh · vigência 2026-02-10 · oficial
   Política de trabalho remoto > 4. Auxílio home office 4.1 A empresa paga um auxílio home office mensal de R$ 120,00, creditado junto com o salário, para ajudar c…
3. [0.0149] rh-jornada-e-banco-de-horas#0  ·  wiki-rh · vigência 2025-09-15 · oficial
   Jornada de trabalho e banco de horas > 1. Jornada 1.1 A jornada padrão é de 8 horas diárias e 40 horas semanais, com 1 hora de intervalo para almoço.

── web (Serper)
Busca na web indisponível: não achei encontro-02/mcp_aurora/busca_web.py acima de C:\Users\cstefano\Desktop\Carlos\FIAP\Cursos\Engenharia de Software\AI Agents Engineering\encontro-03-para-o-repositorio\encontro-03. A pas

In [18]:
pergunta = "O que a CLT diz sobre dividir as férias em mais de um período?"
print("── wiki da Aurora (configuração final)"); r.mostrar(final(pergunta, 3))
print("\n── web (Serper)");                       r.mostrar_web(r.buscar_na_web(pergunta))

── wiki da Aurora (configuração final)
1. [0.0323] rh-ferias#1  ·  wiki-rh · vigência 2025-11-20 · oficial
   Política de férias > 2. Divisão em períodos 2.1 As férias podem ser divididas em até três períodos. Um dos períodos deve ter pelo menos 14 dias corridos, e os o…
2. [0.0323] rh-ferias#0  ·  wiki-rh · vigência 2025-11-20 · oficial
   Política de férias > 1. Direito a férias 1.1 Todo colaborador tem direito a 30 dias de férias a cada 12 meses de trabalho.
3. [0.032] rh-ferias#3  ·  wiki-rh · vigência 2025-11-20 · oficial
   Política de férias > 4. Venda de férias 4.1 É possível vender até 10 dias de férias (abono pecuniário), desde que o pedido seja feito junto com a solicitação da…

── web (Serper)
Busca na web indisponível: não achei encontro-02/mcp_aurora/busca_web.py acima de C:\Users\cstefano\Desktop\Carlos\FIAP\Cursos\Engenharia de Software\AI Agents Engineering\encontro-03-para-o-repositorio\encontro-03. A pasta encontro-03 precisa estar dentro do repositório, ao lado de e

**Pergunta para o grupo.** Na primeira pergunta, os resultados da web são relevantes? Algum deles vale para a Aurora, e como você saberia? Na segunda, qual das duas fontes é a certa? Se o agente tiver as duas ferramentas, o que a descrição de cada uma precisa dizer sobre quando **não** usá-la?

> **Para ir além (opcional).** Leia o contrato de `buscar_na_web` em `encontro-02/mcp_aurora/busca_web.py`. Ele já responde à última pergunta. E o que acontece se uma das páginas da web tiver um texto escrito para o modelo, e não para o leitor? Guarde para o Encontro 6.

## Etapa 8 · O que o LangChain fez, e o que foi decisão sua

| Peça | Quem fez | Quem decidiu |
|---|---|---|
| Cortar texto respeitando parágrafos | `RecursiveCharacterTextSplitter` | você: tamanho e sobreposição |
| Cortar nas seções do markdown | `MarkdownHeaderTextSplitter` | você: em que nível cortar, prefixar o título |
| Transformar texto em vetor | `sentence-transformers` | você: qual modelo, os prefixos `query:`/`passage:` |
| Guardar e buscar vetores | Chroma | você: quais metadados guardar, qual filtro aplicar |
| Busca léxica | `rank-bm25` | você: a tokenização e as palavras ignoradas |
| Juntar vetor e léxico | 15 linhas em `rag_aurora.py` | você: RRF em vez de somar notas |
| Escolher a versão vigente | 6 linhas em `rag_aurora.py` | você: que a vigência manda |

O framework economizou código. Nenhuma decisão do lado direito saiu dele, e são essas decisões que mudaram o placar.

> **Para ir além (opcional).** O mesmo pipeline com as classes prontas do LangChain fica assim (requer `pip install langchain-chroma langchain-huggingface langchain-community`; não é preciso para a aula):
>
> ```python
> from langchain_huggingface import HuggingFaceEmbeddings
> from langchain_chroma import Chroma
> from langchain_community.retrievers import BM25Retriever
> from langchain.retrievers import EnsembleRetriever
>
> vetorial = Chroma.from_documents(chunks_e, HuggingFaceEmbeddings(model_name=emb.nome)).as_retriever(search_kwargs={"k": 3})
> lexico = BM25Retriever.from_documents(chunks_e, k=3)
> hibrido = EnsembleRetriever(retrievers=[vetorial, lexico], weights=[0.5, 0.5])
> ```
>
> São menos linhas. Repare no que sumiu de vista: os prefixos do E5, o filtro por autoridade e a regra de vigência. Continuam sendo decisão sua, só ficaram mais fáceis de esquecer.

## Próximo passo

Até aqui, a busca roda **sempre**. Na demonstração a seguir, a mesma busca vira uma ferramenta, com o contrato do Encontro 2, e o agente decide se e quando usá-la. Depois, no `02-hands-on-chunking.ipynb`, você escolhe a estratégia para o caso da sua equipe e escreve a nota de chunking.